# Random Forest — Bank Health Classification

Tree-ensemble model predicting `bank_condition` (Healthy / Stressed / Critical).

Key properties:

- **Scaling:** not required for Random Forest — no scaler is applied.
- **Split:** scenario-grouped train/test split with a fixed seed, identical to the
  Logistic Regression and XGBoost notebooks, so all three models are comparable.
- **Class imbalance:** `class_weight="balanced"`.
- **Hyperparameters:** kept at the project's original baseline configuration
  (untuned), the same no-tuning policy applied to all three models so the
  comparison stays fair.
- **Metrics:** accuracy, balanced accuracy, macro precision/recall/F1,
  Critical-class recall and ROC-AUC (macro one-vs-rest), all on the test set.

## 1. Data Loading

In [1]:
from pathlib import Path

import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

PROCESSED_DIR = Path("../processed")
MODELS_DIR = Path("../models")
OUTPUT_DIR = Path("../output")

MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
CLASS_ORDER = ["Healthy", "Stressed", "Critical"]
TARGET_MAPPING = {"Healthy": 0, "Stressed": 1, "Critical": 2}

SHOCK_COLS = [
    "gdp_shock_pp",
    "unemp_shock_pp",
    "rate_shock_pp",
    "credit_spread_bps",
    "inflation_shock_pp",
    "fx_devaluation_pct",
]

In [2]:
features = pd.read_csv(PROCESSED_DIR / "bank_health_features.csv")

TARGET = "bank_condition"
DROP_COLS = ["bank_id", "scenario_id", TARGET, "bank_condition_code"]

X = features.drop(columns=DROP_COLS)
y = features[TARGET]
groups = features["scenario_id"]

print("Dataset shape:", features.shape)
print("Number of model features:", X.shape[1])
print("\nTarget distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

Dataset shape: (20000, 18)
Number of model features: 14

Target distribution (%):
bank_condition
Healthy     51.46
Stressed    32.78
Critical    15.76
Name: proportion, dtype: float64


## 2. Preprocessing

### 2.1 Scenario-Grouped Train/Test Split

Grouped by `scenario_id` so no scenario appears in both train and test.
Parameters and seed are identical in all three model notebooks.

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_scenarios = set(groups.iloc[train_idx])
test_scenarios = set(groups.iloc[test_idx])

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Training scenarios:", len(train_scenarios))
print("Testing scenarios :", len(test_scenarios))
print("Scenario overlap  :", train_scenarios & test_scenarios)

Train shape: (16000, 14) | Test shape: (4000, 14)
Training scenarios: 400
Testing scenarios : 100
Scenario overlap  : set()


### 2.2 Leakage-Free `shock_severity_score`

Rebuilt from the raw shock columns with min/max statistics from the **training**
scenarios only (the version in the feature file was scaled on all scenarios
before the split).

In [4]:
panel_shocks = (
    pd.read_csv(
        PROCESSED_DIR / "bank_stress_simulated_panel_clean.csv",
        usecols=["scenario_id"] + SHOCK_COLS,
    )
    .drop_duplicates("scenario_id")
    .set_index("scenario_id")[SHOCK_COLS]
    .abs()
)

row_magnitudes = features[["scenario_id"]].join(panel_shocks, on="scenario_id")[SHOCK_COLS]
assert row_magnitudes.notna().all().all(), "Missing shock values for some scenarios"

train_mins = row_magnitudes.iloc[train_idx].min()
train_maxs = row_magnitudes.iloc[train_idx].max()

shock_severity = ((row_magnitudes - train_mins) / (train_maxs - train_mins)).mean(axis=1)

X_train["shock_severity_score"] = shock_severity.iloc[train_idx].values
X_test["shock_severity_score"] = shock_severity.iloc[test_idx].values

print("shock_severity_score rebuilt using train-only min/max scaling.")
print(X_train["shock_severity_score"].describe().round(4))

shock_severity_score rebuilt using train-only min/max scaling.
count    16000.0000
mean         0.4166
std          0.2356
min          0.0201
25%          0.2099
50%          0.4012
75%          0.6212
max          0.8932
Name: shock_severity_score, dtype: float64


## 3. Training

Pipeline: median imputation -> Random Forest. No scaling (trees are
scale-invariant). Hyperparameters are the project's original baseline values.

In [5]:
random_forest_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                max_features="sqrt",
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

random_forest_model.fit(X_train, y_train)
print("Random Forest training completed.")

Random Forest training completed.


## 4. Evaluation

In [6]:
y_pred = random_forest_model.predict(X_test)

classes = list(random_forest_model.named_steps["classifier"].classes_)
proba = random_forest_model.predict_proba(X_test)[:, [classes.index(c) for c in CLASS_ORDER]]

report = classification_report(y_test, y_pred, labels=CLASS_ORDER, output_dict=True)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(
        y_test, y_pred, average="macro", labels=CLASS_ORDER, zero_division=0
    ),
    "Recall (macro)": recall_score(
        y_test, y_pred, average="macro", labels=CLASS_ORDER, zero_division=0
    ),
    "F1 (macro)": f1_score(
        y_test, y_pred, average="macro", labels=CLASS_ORDER, zero_division=0
    ),
    "Recall (Critical)": report["Critical"]["recall"],
    "ROC-AUC (macro OVR)": roc_auc_score(
        y_test.map(TARGET_MAPPING),
        proba,
        multi_class="ovr",
        average="macro",
        labels=[0, 1, 2],
    ),
}

print(pd.Series(test_metrics).round(4).to_string())

Accuracy               0.8382
Balanced Accuracy      0.8215
Precision (macro)      0.8238
Recall (macro)         0.8215
F1 (macro)             0.8226
Recall (Critical)      0.7935
ROC-AUC (macro OVR)    0.9566


In [7]:
print(classification_report(y_test, y_pred, labels=CLASS_ORDER, digits=4))

confusion = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
confusion_output = pd.DataFrame(
    confusion,
    index=[f"Actual_{c}" for c in CLASS_ORDER],
    columns=[f"Predicted_{c}" for c in CLASS_ORDER],
)
print(confusion_output)

              precision    recall  f1-score   support

     Healthy     0.9046    0.8987    0.9017      1974
    Stressed     0.7578    0.7724    0.7650      1353
    Critical     0.8091    0.7935    0.8012       673

    accuracy                         0.8383      4000
   macro avg     0.8238    0.8215    0.8226      4000
weighted avg     0.8389    0.8383    0.8385      4000

                 Predicted_Healthy  Predicted_Stressed  Predicted_Critical
Actual_Healthy                1774                 198                   2
Actual_Stressed                184                1045                 124
Actual_Critical                  3                 136                 534


In [8]:
importances = pd.Series(
    random_forest_model.named_steps["classifier"].feature_importances_,
    index=X_train.columns,
).sort_values(ascending=False)

print("Random Forest feature importance (impurity-based):")
print(importances.round(4).to_string())

Random Forest feature importance (impurity-based):
shock_severity_score        0.2975
liquidity_buffer            0.2255
sector_risk_score           0.0820
car_buffer                  0.0656
severity_score              0.0560
baseline_roa_pct            0.0549
risk_x_severity             0.0416
deposit_to_asset_ratio      0.0369
loan_to_asset_ratio         0.0345
top_sector_weight           0.0317
bank_risk_factor            0.0282
concentration_x_severity    0.0236
size_score                  0.0144
concentration_flag          0.0075


## 5. Save Artifacts

Saves the fitted pipeline plus the test-set predictions vs actuals and the
metrics table consumed by the model comparison notebook.

In [9]:
model_path = MODELS_DIR / "random_forest_model.pkl"
joblib.dump(random_forest_model, model_path)

predictions_output = pd.DataFrame(
    {
        "bank_id": features.iloc[test_idx]["bank_id"].values,
        "scenario_id": features.iloc[test_idx]["scenario_id"].values,
        "actual_condition": y_test.values,
        "predicted_condition": y_pred,
        "prob_healthy": proba[:, 0],
        "prob_stressed": proba[:, 1],
        "prob_critical": proba[:, 2],
    }
)
predictions_path = OUTPUT_DIR / "random_forest_predictions.csv"
predictions_output.to_csv(predictions_path, index=False)

metrics_output = pd.DataFrame(
    {"Metric": list(test_metrics), "Score": list(test_metrics.values())}
)
metrics_path = OUTPUT_DIR / "random_forest_metrics.csv"
metrics_output.to_csv(metrics_path, index=False)

print("Saved:", model_path)
print("Saved:", predictions_path)
print("Saved:", metrics_path)

Saved: ..\models\random_forest_model.pkl
Saved: ..\output\random_forest_predictions.csv
Saved: ..\output\random_forest_metrics.csv
